In [1]:
#Noise injection + RF + SHAP

# ===============================
# 1. Setup
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import shap

/Users/anurag/miniconda3/envs/shap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===============================
# 2. Load Data
# ===============================
df = pd.read_csv("../dataset/Heart_disease_cleveland_new.csv")

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# ===============================
# 3. Noise Levels
# ===============================
noise_levels = [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80]

results = []
shap_importances = {}

In [7]:
# ===============================
# 4. Loop Over Noise
# ===============================
for noise_level in noise_levels:

    # Add noise
    noise = np.random.normal(0, noise_level, X_train_scaled.shape)
    X_train_noisy = X_train_scaled + noise

    # Train model
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_train_noisy, y_train)

    # Evaluate
    y_pred = rf_model.predict(X_test_scaled)
    y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    print(f"\nNoise Level: {noise_level}")
    print("Accuracy:", acc)
    print("F1:", f1)
    print("ROC-AUC:", roc)

    # SHAP
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_test_scaled)
    shap_values_class1 = shap_values[1]

    mean_abs_shap = np.mean(np.abs(shap_values_class1), axis=0)

    shap_importances[noise_level] = mean_abs_shap

    results.append([noise_level, acc, f1, roc])


Noise Level: 0
Accuracy: 0.8852459016393442
F1: 0.8852459016393442
ROC-AUC: 0.9512987012987013

Noise Level: 0.05
Accuracy: 0.9016393442622951
F1: 0.896551724137931
ROC-AUC: 0.9550865800865801

Noise Level: 0.1
Accuracy: 0.8852459016393442
F1: 0.8813559322033898
ROC-AUC: 0.952922077922078

Noise Level: 0.15
Accuracy: 0.8688524590163934
F1: 0.8620689655172413
ROC-AUC: 0.9567099567099567

Noise Level: 0.2
Accuracy: 0.8852459016393442
F1: 0.8813559322033898
ROC-AUC: 0.9502164502164502

Noise Level: 0.25
Accuracy: 0.9344262295081968
F1: 0.9310344827586207
ROC-AUC: 0.9680735930735931

Noise Level: 0.3
Accuracy: 0.9016393442622951
F1: 0.9
ROC-AUC: 0.9637445887445887

Noise Level: 0.35
Accuracy: 0.9016393442622951
F1: 0.896551724137931
ROC-AUC: 0.9507575757575757

Noise Level: 0.4
Accuracy: 0.9180327868852459
F1: 0.9152542372881356
ROC-AUC: 0.9567099567099567

Noise Level: 0.45
Accuracy: 0.9016393442622951
F1: 0.896551724137931
ROC-AUC: 0.9604978354978355

Noise Level: 0.5
Accuracy: 0.852459

In [8]:
# ===============================
# 5. Results Table
# ===============================
results_df = pd.DataFrame(results, columns=["Noise", "Accuracy", "F1", "ROC-AUC"])
print(results_df)

    Noise  Accuracy        F1   ROC-AUC
0    0.00  0.885246  0.885246  0.951299
1    0.05  0.901639  0.896552  0.955087
2    0.10  0.885246  0.881356  0.952922
3    0.15  0.868852  0.862069  0.956710
4    0.20  0.885246  0.881356  0.950216
5    0.25  0.934426  0.931034  0.968074
6    0.30  0.901639  0.900000  0.963745
7    0.35  0.901639  0.896552  0.950758
8    0.40  0.918033  0.915254  0.956710
9    0.45  0.901639  0.896552  0.960498
10   0.50  0.852459  0.842105  0.941558
11   0.60  0.918033  0.915254  0.972944
12   0.70  0.868852  0.862069  0.943723
13   0.80  0.868852  0.857143  0.941017
